# Practice: making deep nets train
*AI from scratch, toolkit 4* · lesson: https://ihor.sh/ai/making-nets-train.html

Measure a signal dying through 20 layers and save it with He initialization, write dropout yourself, catch the
missing `model.eval()`, then build a residual block and watch it rescue a 30-layer network that can't learn at all.
Runs on a CPU in about a minute. Replace each `...` under `# YOUR CODE HERE`, run the cell, then the check under it;
`hint(n)` and `answer(n)` are there when you're stuck.

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['Multiplying by 0.8 thirty times is a power.', 'left = 0.8 ** 30'], 2: ['He: variance 2 / n_in, so the spread (standard deviation) is its square root.', 'he = math.sqrt(2 / n_in)'], 3: ['mask = (torch.rand_like(h) > p).float() is 1 where an entry survives.', 'return h * mask / (1 - p)'], 4: ["What switches dropout off and makes batch norm use its saved averages? (The lesson's section 2 question.)", 'model.eval() first, then inside torch.no_grad(): preds = model(x3).argmax(1)'], 5: ['self.f = nn.Sequential(nn.Linear(n, n), nn.ReLU(), nn.Linear(n, n))'], 6: ['y = x + f(x)', 'return x + self.f(x)'], 7: ['A net that learns nothing guesses uniformly over 10 digits: loss ln(10).']}
_ANSWERS = {1: 'bGVmdCA9IDAuOCAqKiAzMA==', 2: 'aGUgPSBtYXRoLnNxcnQoMiAvIG5faW4p', 3: 'bWFzayA9ICh0b3JjaC5yYW5kX2xpa2UoaCkgPiBwKS5mbG9hdCgpCnJldHVybiBoICogbWFzayAvICgxIC0gcCk=', 4: 'bW9kZWwuZXZhbCgpCndpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgcHJlZHMgPSBtb2RlbCh4MykuYXJnbWF4KDEp', 5: 'c2VsZi5mID0gbm4uU2VxdWVudGlhbChubi5MaW5lYXIobiwgbiksIG5uLlJlTFUoKSwgbm4uTGluZWFyKG4sIG4pKQ==', 6: 'cmV0dXJuIHggKyBzZWxmLmYoeCk=', 7: 'Z3Vlc3NfcGxhaW5fbG9zcyA9IG1hdGgubG9nKDEwKQ=='}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from sklearn.datasets import load_digits
torch.manual_seed(0)

## 1. Estimate first
If every layer multiplies the signal's spread by 0.8, what's left after 30 layers? Set `left`.

In [ ]:
left = None
# YOUR CODE HERE (stuck? hint(1), then answer(1))
...
print(left)

In [ ]:
check(1, lambda: abs(left - 0.8 ** 30) < 1e-12, "0.8³⁰ ≈ 0.0012: about a thousandth of the signal survives")

## 2. Measure it, then fix it with He initialization
`spreads(std)` pushes a batch through 20 layers of 256 neurons with ReLU, every weight drawn with spread `std`,
and returns the spread after each layer. Set `he` to He's spread for `n_in = 256` inputs: the square root of
`2 / n_in` (`math.sqrt`).

In [ ]:
def spreads(std, n=256, depth=20):
    x = torch.randn(512, n)
    out = []
    for _ in range(depth):
        layer = nn.Linear(n, n, bias=False)
        nn.init.normal_(layer.weight, mean=0.0, std=std)     # set every weight by hand
        x = F.relu(layer(x))
        out.append(x.std().item())
    return out

n_in = 256
he = None
# YOUR CODE HERE (stuck? hint(2), then answer(2))
...

In [ ]:
with torch.no_grad():
    for label, std in [("1/sqrt(n)", 1 / math.sqrt(n_in)), ("He", he)]:
        if std is not None:
            s = spreads(std)
            plt.semilogy(range(1, 21), s, "o-", label=f"{label}: layer 20 spread {s[-1]:.2g}")
plt.xlabel("layer"); plt.ylabel("spread (log scale)"); plt.legend(); plt.show()
check(2, lambda: abs(he - 0.0884) < 1e-3 and 0.3 < spreads(he)[-1] < 2, "√(2/256) ≈ 0.088 keeps the spread steady through all 20 layers")

## 3. Write dropout
Write `dropout(h, p)`: keep each entry with probability `1 - p` and scale the kept ones by `1 / (1 - p)`.
`torch.rand_like(h)` gives random numbers between 0 and 1 in `h`'s shape; `(... > p).float()` turns a comparison
into 0s and 1s.

In [ ]:
def dropout(h, p):
    # YOUR CODE HERE (stuck? hint(3), then answer(3))
    ...

In [ ]:
h = torch.full((100_000,), 2.0)
check(3, lambda: abs(dropout(h, 0.3).mean().item() - 2.0) < 0.03 and abs((dropout(h, 0.3) == 0).float().mean().item() - 0.3) < 0.01,
      "about 30% zeros, and the mean stays 2.0 because survivors become 2 / 0.7")

## 4. Find the bug: the same input, different answers
This model has dropout and batch norm. The "app" below asks it about the same digit three times and gets three
different answers. Fix it with one call before predicting, then set `preds` to the model's three predicted classes
(`.argmax(1)` of the output) for `x3`.

In [ ]:
X, y = load_digits(return_X_y=True)
X = torch.tensor(X / 16, dtype=torch.float32); y = torch.tensor(y)
perm = torch.randperm(len(X), generator=torch.Generator().manual_seed(0))
tr, va = perm[:1400], perm[1400:]
model = nn.Sequential(nn.Linear(64, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.5), nn.Linear(128, 10))
opt = torch.optim.Adam(model.parameters(), lr=1e-2)
for _ in range(150):
    model.train(); opt.zero_grad(); F.cross_entropy(model(X[tr]), y[tr]).backward(); opt.step()
x3 = X[va[:1]].repeat(3, 1)                     # one digit, asked three times
with torch.no_grad():
    print("buggy app:", model(x3).softmax(1).max(1).values)   # different confidences each time: dropout is still on
preds = None
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...
print(preds)

In [ ]:
check(4, lambda: not model.training and len(set(preds.tolist())) == 1 and preds[0].item() == y[va[0]].item(),
      "model.eval(): the same answer three times, and it's the right digit")

## 5. A residual block
Write the class `Residual`: in `__init__`, keep `self.f = nn.Sequential(nn.Linear(n, n), nn.ReLU(), nn.Linear(n, n))`;
in `forward`, return the input plus `self.f` of it. (Classes and `__init__`/`forward`: the training-in-PyTorch lesson.)

In [ ]:
class Residual(nn.Module):
    def __init__(self, n):
        super().__init__()
        # YOUR CODE HERE (stuck? hint(5), then answer(5))
        ...

    def forward(self, x):
        # YOUR CODE HERE (stuck? hint(6), then answer(6))
        ...

In [ ]:
check(6, lambda: Residual(8)(torch.zeros(2, 8)).shape == (2, 8) and sum(p.numel() for p in Residual(8).parameters()) == 2 * (8 * 8 + 8),
      "output shape = input shape; two 8 × 8 layers with biases inside")

## 6. Thirty layers, plain and residual
The same depth, the same optimizer, 100 full-batch steps. **Predict** before running: will the plain stack learn
anything? Set `guess_plain_loss` to the training loss you expect it to end at (hint: the lesson's loss-curve doctor).

In [ ]:
guess_plain_loss = None
# YOUR CODE HERE (stuck? hint(7), then answer(7))
...

In [ ]:
def deep(kind, depth=30, n=64):
    layers = [nn.Linear(64, n), nn.ReLU()]
    for _ in range(depth):
        layers += [Residual(n)] if kind == "residual" else [nn.Linear(n, n), nn.ReLU()]
    return nn.Sequential(*layers, nn.Linear(n, 10))

def run(kind):
    torch.manual_seed(0)
    m = deep(kind)
    opt = torch.optim.Adam(m.parameters(), lr=1e-3)
    losses = []
    for step in range(100):
        opt.zero_grad(); loss = F.cross_entropy(m(X[tr]), y[tr]); loss.backward(); opt.step(); losses.append(loss.item())
    with torch.no_grad():
        return losses, (m(X[va]).argmax(1) == y[va]).float().mean().item()

results = {}
for kind in ["plain", "residual"]:
    try:
        losses, acc = run(kind)
    except Exception as e:                      # a Residual that isn't written yet
        print(kind, "can't train yet:", type(e).__name__); continue
    results[kind] = (losses[-1], acc)
    plt.plot(losses, label=f"{kind}: final loss {losses[-1]:.3f}, val acc {acc:.3f}")
plt.axhline(math.log(10), ls="--", c="g", label="ln 10"); plt.xlabel("step"); plt.ylabel("training loss"); plt.legend(); plt.show()
check(7, lambda: abs(guess_plain_loss - results["plain"][0]) < 0.05 and results["residual"][1] > 0.9,
      "the plain stack sits at ln 10 ≈ 2.30 (nothing reaches the early layers); the residual one passes 90%")

## Next
Back to the path: https://ihor.sh/ai/ · Karpathy, "A Recipe for Training Neural Networks":
https://karpathy.github.io/2019/04/25/recipe/